# 03 · Inventory policy and day-by-day simulation

**Goal.** Turn forecasts into *when* and *how much* to order, and prove on the test period that it beats the
simple rule used today.

| Policy | Rule |
|---|---|
| **P0 · naive weekly** | every Monday order the average weekly demand of the validation period |
| **P1 · static (s, Q)** | order $Q$ when the inventory position drops to $s = \bar d L + z\,\sigma_d\sqrt{L}$ |
| **P2 · forecast-driven (s, Q)** | same, but $s_t = \sum_{k=1}^{L} \hat d_{t+k} + z\,\sigma_E$ from the best forecast |

* Lot size: $Q^* = \sqrt{2DS/h}$ (EOQ), the same for P1 and P2.
* $\sigma_E$ = RMSE of the forecast error summed over the lead time, measured on the validation period.
* Simulation rules: orders arrive after $L$ days; unmet demand is lost in physical stores (80% waits in online
  stores); holding cost is charged on the end-of-day stock.
* **Iso-service comparison:** a *cycle* service level of 95% ($z = 1.645$) is not a 95% *fill rate*. To compare
  fairly, each policy's knob ($z$, or P0's quantity multiplier) is **calibrated per series** until the simulated
  fill rate reaches the target — only then is the stock compared (ADR-0004).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import load_config

cfg = load_config(ROOT / "config" / "params.yaml")
RESULTS = ROOT / "data" / "processed" / "work" / "results.joblib"
FIG = ROOT / "reports" / "figures"
assert RESULTS.exists(), "Run `python -m src.pipeline --all` from the project root first (about 2 minutes)."
res = joblib.load(RESULTS)
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)
print(f"Data: {res.data.file_name} · {res.dm.n_series} series · test period {res.profile.test[0]} → {res.profile.test[1]}")

## 1 · The parameters of one series

In [ ]:
from src.pipeline import example_series
from src.policy import policy_parameters

store, item = example_series(res, cfg.simulation.detail_stores)
row = res.dm.series_index(store, item)
inp = res.policy_inputs.subset([row])
p = policy_parameters(inp, cfg.channel("physical"), cfg)
L = int(p.lead_time[0])
pd.Series({
    "mean daily demand d̄ (validation)": inp.mean_daily[0],
    "std of daily demand σ_d": inp.sigma_daily[0],
    "lead time L (days)": L,
    "σ over the lead time, P1 (σ_d·√L)": p.sigma_L_hist[0],
    "σ of the forecast error over L, P2": p.sigma_L_fc[0],
    "z for a 95% cycle service level": p.z_p1[0],
    "safety stock P1": p.z_p1[0] * p.sigma_L_hist[0],
    "safety stock P2": p.z_p2[0] * p.sigma_L_fc[0],
    "reorder point P1": inp.mean_daily[0] * L + p.z_p1[0] * p.sigma_L_hist[0],
    "average reorder point P2": p.ltd[0].mean() + p.z_p2[0] * p.sigma_L_fc[0],
    "EOQ lot Q": p.Q[0],
}, name=f"store {store}, item {item}").round(2)

## 2 · With z set in advance (95% cycle service level)

In [ ]:
from src.kpi import aggregate_kpis

phys = res.summary[res.summary["channel"] == "physical"]
cols = ["policy", "fill_rate", "cycle_service_level", "avg_inventory_value", "stockout_days", "n_orders", "total_cost"]
aggregate_kpis(phys[phys["variant"] == "fixed"], ["policy"])[cols]

With big EOQ lots, a 95% *cycle* service level already gives a fill rate close to 100%: the (s, Q) policies
hold more stock than a 95% fill rate needs. The fair comparison calibrates every policy to the same fill rate.

## 3 · Same fill rate, less stock

In [ ]:
cal = aggregate_kpis(phys[phys["variant"] == "calibrated"], ["policy"])
display(cal[cols])
display(Image(FIG / "policy_inventory_at_target.png"))
display(Image(FIG / "policy_tradeoff.png"))

In [ ]:
display(Image(FIG / "cost_breakdown.png"))
display(Image(FIG / "cost_vs_service.png"))

## 4 · Day by day

In [ ]:
display(Image(FIG / "inventory_trajectory.png"))

## 5 · Sensitivity and channels

In [ ]:
display(Image(FIG / "sensitivity.png"))
res.sensitivity.pivot_table(index=["lead_time", "service_level"], columns="policy", values="avg_inventory_value").round(0)

In [ ]:
display(Image(FIG / "channel_comparison.png"))
aggregate_kpis(res.summary[res.summary["variant"] == "calibrated"], ["channel", "policy"])[
    ["channel", "policy", "fill_rate", "avg_inventory_value", "total_cost"]]

## 6 · Any scenario: stores × products × channel

A scenario only filters and sums the per-series results, so it is instant and exact (ADR-0005). Example: the
first 5 stores, the 20 top-revenue items, stores 4 and 5 sold online.

In [ ]:
from src.scenario import apply_scenario, build_scenario
from src.series import natural_sorted

stores = natural_sorted(res.data.stores["store"])
sc = build_scenario(stores, res.classification.set_index("item")["revenue"], n_stores=min(5, len(stores)),
                    n_items=min(20, len(res.classification)), online_stores=stores[3:5])
print(sc.label)
k = aggregate_kpis(apply_scenario(res.summary, sc).query("variant == 'calibrated'"), ["policy"])
k[["policy", "fill_rate", "avg_inventory_value", "total_cost"]]

## 7 · Findings

In [ ]:
h = res.headline["inventory"]
hf = res.headline["inventory_fixed_z"]
best_cost = res.tradeoff.loc[res.tradeoff["total_cost"].idxmin()]
display(Markdown(f"""
* At the same **{res.headline['target_fill_rate']:.0%} fill rate**, the forecast-driven policy needs
  **{h['inventory_reduction_pct']:.0%} less average inventory** than the naive weekly order
  ({h['avg_inventory_value_candidate']:,.0f} € vs {h['avg_inventory_value_baseline']:,.0f} €) and
  **{h['inventory_reduction_vs_classic_pct']:.0%} less** than the classic static (s, Q) policy.
* Total cost falls by **{h['total_cost_reduction_pct']:.0%}** vs the naive order
  (€{h['total_cost_reduction_eur']:,.0f} over the test period).
* With z set in advance (95% *cycle* service) the (s, Q) policies reach a fill rate of
  {hf['fill_rate_candidate']:.1%} (P2): the cycle service level understates the real service.
* Under the assumed costs a lost sale costs far more than a year of holding, so total cost keeps falling up to
  a fill rate of about **{best_cost['fill_rate']:.1%}**: 95% is a conservative target, and the forecast-driven
  policy stays the cheapest across the whole curve.
"""))